<a href="https://colab.research.google.com/github/Dires-colab-pro/Monkeypox-disease-stages-dataset/blob/main/Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_DIR = "/content/drive/MyDrive/dataset"
NUM_CLASSES = 6
CLASS_NAMES = ["Macules", "Papules", "Vesicles", "Pustules", "Scabs", "Normal"]


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(25),
    transforms.ColorJitter(0.3,0.3,0.3),
    transforms.RandomAffine(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])


In [ ]:
train_ds = datasets.ImageFolder(DATA_DIR+"/train", transform=train_transform)
val_ds   = datasets.ImageFolder(DATA_DIR+"/val", transform=test_transform)
test_ds  = datasets.ImageFolder(DATA_DIR+"/test", transform=test_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)


In [ ]:
class MpoxCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(256*14*14,512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512,NUM_CLASSES)
        )

    def forward(self,x):
        x = self.features(x)
        x = x.view(x.size(0),-1)
        return self.classifier(x)


In [ ]:
def train_model(model, epochs=25):
    model.to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()

    tr_acc, val_acc, tr_loss, val_loss = [],[],[],[]

    for ep in range(epochs):
        model.train()
        c,t,l = 0,0,0

        for x,y in train_loader:
            x,y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out,y)
            loss.backward()
            optimizer.step()

            l += loss.item()
            c += (out.argmax(1)==y).sum().item()
            t += y.size(0)

        model.eval()
        vc,vt,vl = 0,0,0
        with torch.no_grad():
            for x,y in val_loader:
                out = model(x.to(DEVICE))
                vl += criterion(out,y.to(DEVICE)).item()
                vc += (out.argmax(1)==y.to(DEVICE)).sum().item()
                vt += y.size(0)

        tr_acc.append(c/t)
        val_acc.append(vc/vt)
        tr_loss.append(l/len(train_loader))
        val_loss.append(vl/len(val_loader))

        print(f"Epoch {ep+1}: TrainAcc={tr_acc[-1]:.4f}, ValAcc={val_acc[-1]:.4f}")

    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.plot(tr_acc,label="Train")
    plt.plot(val_acc,label="Val")
    plt.title("Accuracy")
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(tr_loss,label="Train")
    plt.plot(val_loss,label="Val")
    plt.title("Loss")
    plt.legend()
    plt.show()

    return model


In [ ]:
def evaluate(model):
    model.eval()
    y_true, y_pred = [],[]

    with torch.no_grad():
        for x,y in test_loader:
            out = model(x.to(DEVICE))
            y_pred.extend(out.argmax(1).cpu().numpy())
            y_true.extend(y.numpy())

    print("Accuracy :", accuracy_score(y_true,y_pred))
    print("Precision:", precision_score(y_true,y_pred,average="macro"))
    print("Recall   :", recall_score(y_true,y_pred,average="macro"))
    print("F1-Score :", f1_score(y_true,y_pred,average="macro"))

    cm = confusion_matrix(y_true,y_pred)
    cm = cm / cm.sum(axis=1,keepdims=True)

    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, cmap="Blues",
                xticklabels=CLASS_NAMES,
                yticklabels=CLASS_NAMES,
                fmt=".2f")
    plt.title("Normalized Confusion Matrix")
    plt.show()


In [ ]:
class GradCAM:
    def __init__(self, model, layer):
        self.model = model
        self.grad = None
        layer.register_backward_hook(self.save_grad)

    def save_grad(self, m, gi, go):
        self.grad = go[0]

    def generate(self, x, cls):
        feats = self.model.features(x)
        out = self.model(x)
        self.model.zero_grad()
        out[0,cls].backward()

        weights = self.grad.mean((2,3), keepdim=True)
        cam = torch.relu((weights * feats).sum(1))
        cam = cam.squeeze().cpu().numpy()
        cam = cv2.resize(cam,(224,224))
        cam = cam / cam.max()
        return cam


In [ ]:
def show_gradcam(img_path):
    model.eval()

    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb,(224,224))

    input_tensor = test_transform(
        transforms.ToPILImage()(img_resized)
    ).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        out = model(input_tensor)
        probs = torch.softmax(out,1).cpu().numpy()[0]
        pred = probs.argmax()

    cam = gradcam.generate(input_tensor, pred)
    heatmap = cv2.applyColorMap(np.uint8(255*cam), cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(img_resized,0.6,heatmap,0.4,0)

    plt.figure(figsize=(15,4))
    plt.subplot(1,4,1)
    plt.imshow(img_resized); plt.title("Original"); plt.axis("off")

    plt.subplot(1,4,2)
    plt.imshow(cam,cmap="jet"); plt.title("Grad-CAM"); plt.axis("off")

    plt.subplot(1,4,3)
    plt.imshow(overlay); plt.title("Overlay"); plt.axis("off")

    plt.subplot(1,4,4)
    plt.bar(CLASS_NAMES, probs)
    plt.xticks(rotation=45)
    plt.title(f"Predicted: {CLASS_NAMES[pred]}")

    plt.show()


In [ ]:
model = MpoxCNN()
model = train_model(model, epochs=10)
evaluate(model)

gradcam = GradCAM(model, model.features[-3])


In [ ]:
show_gradcam("/content/drive/MyDrive/dataset/test/1_Macules/Macules33.PNG")
